In [1]:
import pandas as pd

df=pd.read_csv('/Applications/support-ticket-mlops/data/dataset-tickets-multi-lang-4-20k.csv')

In [3]:
df.head()

,subject,body,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Unvorhergesehener Absturz der Datenanalyse-Pla...,Die Datenanalyse-Plattform brach unerwartet ab...,Ich werde Ihnen bei der Lösung des Problems he...,Incident,General Inquiry,low,de,Crash,Technical,Bug,Hardware,Resolution,Outage,Documentation,NaN
1,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,Request,Customer Service,medium,en,Feedback,Sales,IT,Tech Support,NaN,NaN,NaN,NaN
2,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,Request,Customer Service,medium,en,Technical,Product,Guidance,Documentation,Performance,Feature,NaN,NaN
3,Krankenhaus-Dienstleistung-Problem,Ein Medien-Daten-Sperrverhalten trat aufgrund ...,Zurück zur E-Mail-Beschwerde über den Sperrver...,Incident,Customer Service,high,de,Security,Breach,Login,Maintenance,Incident,Resolution,Feedback,NaN
4,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",Request,Customer Service,medium,en,Security,Customer,Compliance,Breach,Documentation,Guidance,NaN,NaN


In [4]:
print(df.shape)
print(df['language'].value_counts())
print(df['priority'].value_counts())


(20000, 15)
language
en    11923
de     8077
Name: count, dtype: int64
priority
medium    8144
high      7801
low       4055
Name: count, dtype: int64


In [5]:
df_en = df[df['language']=='en'].copy()
print(df_en.shape)
print(df_en['priority'].value_counts()) 

(11923, 15)
priority
medium    4952
high      4571
low       2400
Name: count, dtype: int64


In [6]:
pd.set_option('display.max_colwidth', 300)
df_en[['subject', 'body', 'priority', 'type', 'queue']].sample(5, random_state=1)

,subject,body,priority,type,queue
19890,Expansion of Digital Campaign,Seek assistance with new tools,low,Change,Technical Support
18855,Support for Smart Licht,Seeking detailed guidance on integrating Smart Licht with MacOS Monterey for project management workflows. Could you provide step-by-step instructions and any available resources to help get started? Appreciate information on compatibility and potential issues.,high,Request,Technical Support
19400,Problem with Modem Disconnection,"Hello Customer Support, I am reaching out to report an issue with the frequent disconnection of my modem, which is causing disruptions in secure data transmission. The problem may have arisen due to network overload or outdated firmware. Despite attempting to restart the modem and checking the c...",medium,Incident,IT Support
8687,Concern with Excel 2021 Crashing During Data Analysis,"Hello Customer Support, I have been encountering problems with Excel 2021 crashing while performing data analysis. This issue could be related to incompatible add-ins or outdated drivers. Despite restarting my system, updating Excel, and verifying my dependencies, the problem continues. I would ...",high,Incident,Technical Support
601,Enquiry on Data Security Measures for Hospital Products,"I am seeking detailed information on the data security measures implemented for the services and products offered by your hospital. Could you provide a comprehensive overview of the protocols and technologies used to protect sensitive patient data, including details on encryption, access control...",medium,Request,Customer Service


In [8]:
print(df_en['type'].value_counts())
df_en[['tag_1','tag_2','tag_3']].sample(10, random_state=1)


type
Incident    4642
Request     3498
Problem     2498
Change      1285
Name: count, dtype: int64


,tag_1,tag_2,tag_3
19890,Technical,Campaign,Guidance
18855,Feature,Documentation,Tech Support
19400,Outage,Network,Disruption
8687,Crash,Product,Bug
601,Security,Product,Documentation
19335,Feedback,Sales,Feature
11427,Technical,Crash,Server
8383,Feedback,Sales,IT
3811,Technical,Product,Digital
8110,Bug,Performance,Documentation


In [17]:
severe_tags = {'Outage', 'Crash', 'Bug', 'Security', 'Disruption', 'Network', 'Server', 'Data Loss'}

def compute_escalation_risk(row):
    score = 0
    
    # Signal 1: ticket type — Incident/Problem are more severe than Request/Change
    if row['type'] in ['Incident', 'Problem']:
        score += 1
    
    # Signal 2: existing priority label
    if row['priority'] == 'high':
        score += 1
    
    # Signal 3: severity tags present anywhere in tag_1 to tag_8
    tags = [str(row.get(f'tag_{i}', '')) for i in range(1, 9)]
    if any(tag in severe_tags for tag in tags):
        score += 1
    
    # Signal 4: longer/more complex ticket body
    if len(str(row['body'])) > 500:
        score += 1
    
    return 1 if score >= 3 else 0

df_en['escalated'] = df_en.apply(compute_escalation_risk, axis=1)
df_en['escalated'].value_counts(normalize=True)

escalated
0    0.683972
1    0.316028
Name: proportion, dtype: float64

In [12]:
severe_tags = {'Outage', 'Crash', 'Bug', 'Security', 'Disruption', 'Network', 'Server', 'Data Loss'}

def compute_escalation_risk(row):
    score = 0
    
    # Signal 1: ticket type — Incident/Problem are more severe than Request/Change
    if row['type'] in ['Incident', 'Problem']:
        score += 1
    
    # Signal 2: existing priority label
    if row['priority'] == 'high':
        score += 1
    
    # Signal 3: severity tags present anywhere in tag_1 to tag_8
    tags = [str(row.get(f'tag_{i}', '')) for i in range(1, 9)]
    if any(tag in severe_tags for tag in tags):
        score += 1
    
    # Signal 4: longer/more complex ticket body
    if len(str(row['body'])) > 500:
        score += 1
    
    return 1 if score >= 4 else 0

df_en['escalated'] = df_en.apply(compute_escalation_risk, axis=1)
df_en['escalated'].value_counts(normalize=True)

escalated
0    0.940535
1    0.059465
Name: proportion, dtype: float64

In [18]:
def compute_escalation_risk(row):
    is_severe_type = row['type'] in ['Incident', 'Problem']
    is_high_priority = row['priority'] == 'high'
    tags = [str(row.get(f'tag_{i}', '')) for i in range(1, 9)]
    has_severe_tag = any(tag in severe_tags for tag in tags)
    
    return int(is_severe_type and (is_high_priority or has_severe_tag))

df_en['escalated'] = df_en.apply(compute_escalation_risk, axis=1)
df_en['escalated'].value_counts(normalize=True)

escalated
1    0.519919
0    0.480081
Name: proportion, dtype: float64

In [19]:
def compute_escalation_risk(row):
    is_severe_type = row['type'] in ['Incident', 'Problem']
    is_high_priority = row['priority'] == 'high'
    tags = [str(row.get(f'tag_{i}', '')) for i in range(1, 9)]
    has_severe_tag = any(tag in severe_tags for tag in tags)
    
    # now requires ALL three conditions together
    return int(is_severe_type and is_high_priority and has_severe_tag)

df_en['escalated'] = df_en.apply(compute_escalation_risk, axis=1)
df_en['escalated'].value_counts(normalize=True)

escalated
0    0.786463
1    0.213537
Name: proportion, dtype: float64

In [20]:
df_en['escalated'].value_counts()

escalated
0    9377
1    2546
Name: count, dtype: int64